In [3]:
import torch
from torch import nn
from d2l import torch as d2l
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.optim as optim

#### Without dropout

In [4]:
## THIS CODE IS TO CREATE THE TENSOR FOR THE TRAINING DATA ##

changi_df = pd.read_csv('../../Data/Final/Imputation/imp_train_set.csv')

# Pivot to create a time series for each (x, y)
changi_pivot = changi_df.pivot(index=['x', 'y'], columns='Date', values='Value')
#print (changi_pivot)

# Reset index to keep (x, y) as columns
changi_pivot = changi_pivot.reset_index()

# Convert time columns back into a NumPy array
ts_changi = changi_pivot.iloc[:, 2:].values  # Ignore first two columns (x, y)
loc_changi = changi_pivot.iloc[:, :2].values  # Store coordinates

# Standardization with Z-score normalisation
changi_mean_LST = ts_changi.mean()
changi_std_LST = ts_changi.std()
ts_changi = (ts_changi - changi_mean_LST) / changi_std_LST 

# Convert to PyTorch tensor wiith extra dimension for LST
X_train_tensor_changi = torch.tensor(ts_changi, dtype=torch.float32).unsqueeze(-1)
print(X_train_tensor_changi.shape)

X_train_changi = X_train_tensor_changi
print(X_train_changi)
y_train_changi = X_train_tensor_changi[:, 1:, :]

torch.Size([5290, 137, 1])
tensor([[[ 0.9390],
         [ 1.0576],
         [ 0.9732],
         ...,
         [-0.9473],
         [-0.2548],
         [ 0.0025]],

        [[ 0.9763],
         [ 1.1098],
         [ 0.9292],
         ...,
         [-0.8113],
         [-0.1934],
         [ 0.0646]],

        [[ 0.7565],
         [ 0.9377],
         [ 0.9457],
         ...,
         [-0.7762],
         [-0.1202],
         [ 0.1013]],

        ...,

        [[ 0.1450],
         [ 0.6744],
         [ 0.7059],
         ...,
         [-0.8501],
         [-0.5962],
         [ 0.0429]],

        [[-0.0291],
         [ 0.6774],
         [ 0.7078],
         ...,
         [-0.9023],
         [-0.5899],
         [ 0.0784]],

        [[-0.0341],
         [ 0.3083],
         [ 0.2423],
         ...,
         [-1.0462],
         [-0.6638],
         [-0.0891]]])


In [5]:
## THIS CODE IS TO DEFINE THE LSTM MODEL ##

class LSTMPredictor(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=3, output_size=1):
        super(LSTMPredictor, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)  # Fully connected layer

    def forward(self, x):
        lstm_out, _ = self.lstm(x)  # LSTM output
        out = self.fc(lstm_out)  # Fully connected layer for final prediction
        return out

In [6]:
## THIS CODE IS TO TRAIN THE MODEL ##
torch.manual_seed(5188)

model = LSTMPredictor() # Model
criterion = nn.MSELoss() # Loss function to track performance over epochs
optimizer = optim.Adam(model.parameters(), lr=0.001) # Adam optimizer (can be switched)

# Training loop
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    predictions = model(X_train_changi)

    loss = criterion(predictions[:, :-1, :], y_train_changi)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0: # Print loss every 10 epochs
        print(f"Epoch {epoch}/{num_epochs}, Loss: {loss.item():.4f}")

Epoch 0/100, Loss: 1.0082
Epoch 10/100, Loss: 0.9866
Epoch 20/100, Loss: 0.9414
Epoch 30/100, Loss: 0.9111
Epoch 40/100, Loss: 0.9006
Epoch 50/100, Loss: 0.8959
Epoch 60/100, Loss: 0.8907
Epoch 70/100, Loss: 0.8838
Epoch 80/100, Loss: 0.8766
Epoch 90/100, Loss: 0.8602


In [7]:
## THIS CODE IS FOR FORECASTING##

model.eval()
torch.manual_seed(5188)

with torch.no_grad():
    X_pred_changi = model(X_train_changi)  # Forecast next steps
    X_pred_changi= X_pred_changi[:, -12:, :]  # Extract last 12 bimonthly periods (2023-2024)

# Convert predictions to a NumPy array
X_pred_changi_np = X_pred_changi.squeeze().cpu().numpy()

# **Denormalize predictions**
X_pred_changi_np = (X_pred_changi_np * changi_std_LST) + changi_mean_LST  # Convert back to original from scaled values

# Define bimonthly periods
bimonthly_periods = [
    "Jan-Feb 2023", "Mar-Apr 2023", "May-Jun 2023", "Jul-Aug 2023", "Sep-Oct 2023", "Nov-Dec 2023",
    "Jan-Feb 2024", "Mar-Apr 2024", "May-Jun 2024", "Jul-Aug 2024", "Sep-Oct 2024", "Nov-Dec 2024"
]

# Create a long-format DataFrame
changi_pred_df = pd.DataFrame({
    "x": np.repeat(loc_changi[:, 0], len(bimonthly_periods)),  # Use stored locations
    "y": np.repeat(loc_changi[:, 1], len(bimonthly_periods)),  
    "Date": bimonthly_periods * len(loc_changi),
    "Predicted_LST": X_pred_changi_np.flatten()  # Store denormalized values
})

changi_pred_df.to_csv('changi_pred_long.csv', index=False) # Save to CSV for RMSE

In [ ]:
## THIS CODE IS TO FIND THE RMSE BETWEEN TRUE AND PREDICTED LST ##

true_df = pd.read_csv("changi_pred_long.csv")
pred_df = pd.read_csv("../../Data/Final/TT Split/changi_test_long.csv")

merged_df = true_df.merge(pred_df, on=["x", "y", "Date"], suffixes=("_true", "_pred"))
print(merged_df.head())

# Compute RMSE
rmse = np.sqrt(np.mean((merged_df["Predicted_LST"] - merged_df["Value"]) ** 2))
print(f"The average RMSE between true and predicted LST for Changi is: {rmse:.4f}")

# Filter data for every third step
filtered_df = merged_df[merged_df["Date"].isin([
    "Jan-Feb 2023", "May-Jun 2023", "Nov-Dec 2023", "May-Jun 2024", "Nov-Dec 2024"
])]

# Define the order of selected dates
selected_dates = ["Jan-Feb 2023", "May-Jun 2023", "Nov-Dec 2023", "May-Jun 2024", "Nov-Dec 2024"]

# Group by coordinates and compute RMSE for each selected date
rmse_table = filtered_df.groupby(["x", "y", "Date"]).apply(
    lambda group: np.sqrt(np.mean((group["Predicted_LST"] - group["Value"]) ** 2))
).reset_index(name="RMSE")

# Pivot the table to have dates as columns
rmse_pivot = rmse_table.pivot(index=["x", "y"], columns="Date", values="RMSE")

# Ensure the columns are in the correct order
rmse_pivot = rmse_pivot.reindex(columns=selected_dates)

# Compute the average RMSE across all selected dates
rmse_pivot["Average_RMSE"] = rmse_pivot.mean(axis=1)

# Reset index for a clean table
rmse_pivot = rmse_pivot.reset_index()

# Rename columns for clarity
rmse_pivot.columns.name = None
rmse_pivot = rmse_pivot.rename(columns={
    "Jan-Feb 2023": "f_1",
    "May-Jun 2023": "f_3",
    "Nov-Dec 2023": "f_4",
    "May-Jun 2024": "f_5",
    "Nov-Dec 2024": "f_6",
    "Average_RMSE": "Avg_RMSE"
})

#print(rmse_pivot)

# Save the table to a CSV file
rmse_pivot.to_csv("rmse_table.csv", index=False)

            x         y          Date  Predicted_LST      Value
0  103.964566  1.350459  Jan-Feb 2023      24.993717  19.300148
1  103.964566  1.350459  Mar-Apr 2023      25.426064  21.802744
2  103.964566  1.350459  May-Jun 2023      24.788220  23.389268
3  103.964566  1.350459  Jul-Aug 2023      25.540777  22.872917
4  103.964566  1.350459  Sep-Oct 2023      25.308475  27.238957
RMSE between true and predicted LST for Changi is: 2.9294
               x         y   RMSE_f1   RMSE_f2   RMSE_f3   RMSE_f4   RMSE_f5  \
0     103.964566  1.350459  5.693569  1.398952  4.529729  1.068684  4.406015   
1     103.964566  1.351269  5.741188  1.179699  4.246012  0.973368  4.921117   
2     103.964566  1.352079  5.530923  1.013962  4.159290  0.849487  4.998781   
3     103.964566  1.352889  5.671045  1.088323  4.229623  1.046720  5.011903   
4     103.965377  1.348838  5.704842  1.543901  4.469410  1.390419  4.310287   
...          ...       ...       ...       ...       ...       ...       ...  

#### With dropout

In [ ]:
class LSTMPredictor(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=3, output_size=1, dropout=0.2):
        super(LSTMPredictor, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.dropout = nn.Dropout(dropout)  # Dropout applied after LSTM
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)  # lstm_out: (batch, seq_len, hidden_size)
        lstm_out = lstm_out[:, -1, :]  # Take last time step's output (batch, hidden_size)
        lstm_out = self.dropout(lstm_out)  # Apply dropout before FC layer
        out = self.fc(lstm_out)  # Pass through fully connected layer
        return out  # Shape: (batch, output_size)

In [24]:
# Initialize model
torch.manual_seed(5188)
model = LSTMPredictor()

# Define loss function and optimizer with weight decay
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-8)  # Added weight decay

# Training loop
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    predictions = model(X_train_changi)
    loss = criterion(predictions, y_train_changi)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}/{num_epochs}, Loss: {loss.item():.4f}")

c:\Users\naomi\anaconda3\envs\d2l\lib\site-packages\torch\nn\modules\loss.py:610: UserWarning: Using a target size (torch.Size([5290, 136, 1])) that is different to the input size (torch.Size([5290, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


RuntimeError: The size of tensor a (5290) must match the size of tensor b (136) at non-singleton dimension 1

In [14]:
# Forecasting
model.eval()
torch.manual_seed(5188)
with torch.no_grad():
    X_pred_changi = model(X_train_changi)
    X_pred_changi = X_pred_changi[:, -12:, :]

# Denormalize predictions
X_pred_changi_np = (X_pred_changi.squeeze().cpu().numpy() * changi_std_LST) + changi_mean_LST

# Define bimonthly periods
bimonthly_periods = [
    "Jan-Feb 2023", "Mar-Apr 2023", "May-Jun 2023", "Jul-Aug 2023", "Sep-Oct 2023", "Nov-Dec 2023",
    "Jan-Feb 2024", "Mar-Apr 2024", "May-Jun 2024", "Jul-Aug 2024", "Sep-Oct 2024", "Nov-Dec 2024"
]

# Create DataFrame for predictions
changi_pred_df = pd.DataFrame({
    "x": np.repeat(loc_changi[:, 0], len(bimonthly_periods)),
    "y": np.repeat(loc_changi[:, 1], len(bimonthly_periods)),  
    "Date": bimonthly_periods * len(loc_changi),
    "Predicted_LST": X_pred_changi_np.flatten()
})

# Save predictions to CSV
changi_pred_df.to_csv('changi_pred_long_do.csv', index=False)

In [15]:
## THIS CODE IS TO FIND THE RMSE BETWEEN TRUE AND PREDICTED LST ##

true_df = pd.read_csv("../../Data/Final/Imputation/imp_test_set.csv")
pred_df = pd.read_csv("changi_pred_long_do.csv")

print(true_df.head())
print(pred_df.head())

merged_df = true_df.merge(pred_df, on=["x", "y", "Date"], suffixes=("_true", "_pred"))
print(merged_df.head())

# Compute RMSE
rmse = np.sqrt(np.mean((merged_df["Value"] - merged_df["Predicted_LST"]) ** 2))
print(f"The average RMSE between true and predicted LST for Changi is: {rmse:.4f}")

# Filter data for every third step
filtered_df = merged_df[merged_df["Date"].isin([
    "Jan-Feb 2023", "May-Jun 2023", "Nov-Dec 2023", "May-Jun 2024", "Nov-Dec 2024"
])]

# Define the order of selected dates
selected_dates = ["Jan-Feb 2023", "May-Jun 2023", "Nov-Dec 2023", "May-Jun 2024", "Nov-Dec 2024"]

# Group by coordinates and compute RMSE for each selected date
rmse_table = filtered_df.groupby(["x", "y", "Date"]).apply(
    lambda group: np.sqrt(np.mean((group["Predicted_LST"] - group["Value"]) ** 2))
).reset_index(name="RMSE")

# Pivot the table to have dates as columns
rmse_pivot = rmse_table.pivot(index=["x", "y"], columns="Date", values="RMSE")

# Ensure the columns are in the correct order
rmse_pivot = rmse_pivot.reindex(columns=selected_dates)

# Compute the average RMSE across all selected dates
rmse_pivot["Average_RMSE"] = rmse_pivot.mean(axis=1)

# Reset index for a clean table
rmse_pivot = rmse_pivot.reset_index()

# Rename columns for clarity
rmse_pivot.columns.name = None
rmse_pivot = rmse_pivot.rename(columns={
    "Jan-Feb 2023": "f_1",
    "May-Jun 2023": "f_3",
    "Nov-Dec 2023": "f_4",
    "May-Jun 2024": "f_5",
    "Nov-Dec 2024": "f_6",
    "Average_RMSE": "Avg_RMSE"
})

#print(rmse_pivot)

# Save the table to a CSV file
rmse_pivot.to_csv("rmse_table_dropout.csv", index=False)

            x         y          Date      Value
0  103.964566  1.350459  Jan-Feb 2023  19.300148
1  103.964566  1.351269  Jan-Feb 2023  19.300060
2  103.964566  1.352079  Jan-Feb 2023  19.299971
3  103.964566  1.352889  Jan-Feb 2023  19.299883
4  103.965377  1.348838  Jan-Feb 2023  19.300281
            x         y          Date  Predicted_LST
0  103.964566  1.350459  Jan-Feb 2023      25.661110
1  103.964566  1.350459  Mar-Apr 2023      25.753454
2  103.964566  1.350459  May-Jun 2023      25.695870
3  103.964566  1.350459  Jul-Aug 2023      25.737650
4  103.964566  1.350459  Sep-Oct 2023      25.763147
            x         y          Date      Value  Predicted_LST
0  103.964566  1.350459  Jan-Feb 2023  19.300148      25.661110
1  103.964566  1.351269  Jan-Feb 2023  19.300060      25.775122
2  103.964566  1.352079  Jan-Feb 2023  19.299971      25.650265
3  103.964566  1.352889  Jan-Feb 2023  19.299883      25.705606
4  103.965377  1.348838  Jan-Feb 2023  19.300281      25.944900
The 